# PitchLens — exploration initiale des datasets

Ce notebook est le point d’entrée pour comprendre les données avant de les transformer ou d’entraîner un modèle. Il répond à quatre questions :

1. Quelles sources avons-nous et sous quelles conditions pouvons-nous les utiliser ?
2. Les annotations sont-elles techniquement valides ?
3. Quelle taille aura réellement le ballon après un redimensionnement à 640 px ?
4. Peut-on faire confiance aux splits train/validation/test existants ?

Les cellules appellent les fonctions testées de `src/pitchlens/data`. Le notebook conserve l’explication, les graphiques et les décisions. Il ne modifie jamais les données brutes.

## 0. Préparation

La cellule suivante retrouve la racine du projet, même si le notebook est ouvert depuis le dossier `notebooks/`, puis rend le package local importable.

In [ ]:
from pathlib import Path
import json
import sys
import tomllib

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pitchlens.data.audit_roboflow_splits import audit_splits, parent_name
from pitchlens.data.audit_sequences import audit_sequences
from pitchlens.data.audit_yolo import audit, label_path_for

INFACTORY_ROOT = PROJECT_ROOT / "data/raw/infactory_ball_tracking/data"
SOCCEREYE_ROOT = PROJECT_ROOT / "data/interim/soccereye"

print(f"Projet : {PROJECT_ROOT}")
print(f"Infactory disponible : {INFACTORY_ROOT.is_dir()}")
print(f"SoccerEye disponible : {SOCCEREYE_ROOT.is_dir()}")

## 1. Inventaire des sources

Le registre ne contient pas les images. Il enregistre leur origine, leur licence et leur état dans le projet. Une licence non commerciale est acceptable pour les expériences actuelles ; une licence absente reste un point à clarifier avant toute republication.

In [ ]:
registry_path = PROJECT_ROOT / "configs/data_sources.toml"
with registry_path.open("rb") as stream:
    sources = tomllib.load(stream)["sources"]

rows = ["| Source | État | Licence | Domaine |", "|---|---|---|---|"]
for source in sources:
    rows.append(
        f"| {source['name']} | `{source['status']}` | "
        f"{source['data_license']} | {source['domain']} |"
    )
display(Markdown("\n".join(rows)))

### Origine des deux datasets présents localement

Un dataset ne se résume pas à ses images : il faut pouvoir retrouver son producteur, la page d’origine, la version exacte et les conditions d’utilisation. Les informations ci-dessous proviennent des manifestes enregistrés lors du téléchargement.

In [ ]:
manifest_directory = PROJECT_ROOT / "data/manifests"
infactory_manifest = json.loads(
    (manifest_directory / "infactory_ball_tracking.json").read_text()
)
soccereye_manifest = json.loads(
    (manifest_directory / "soccereye.json").read_text()
)

display(Markdown(f"""
| | Infactory Ball Tracking | SoccerEye |
|---|---|---|
| Producteur | Infactory AI | WWandP / dépôt SoccerEye |
| Page d’origine | [Hugging Face]({infactory_manifest['source_url']}) | [GitHub](https://github.com/WWandP/SoccerEye) puis Google Drive |
| Type de vues | Retransmissions professionnelles, plans larges et gros plans | Vues professionnelles/scouting, parfois tuilées |
| Annotation | Une classe : ballon ; frames sans fichier TXT = négatifs | Deux classes : ballon et personne |
| Version locale | Commit `{infactory_manifest['resolved_revision'][:12]}` | Archive SHA-256 `{soccereye_manifest['archive_sha256'][:12]}…` |
| Licence retenue | CC BY-NC 4.0 | Licence des données non précisée |
| Rôle PitchLens | Base temporelle initiale | Complément pour ballons minuscules |
"""))

print("Données brutes Infactory :", INFACTORY_ROOT)
print("Données extraites SoccerEye :", SOCCEREYE_ROOT)

## 2. Infactory : contenu réel

`audit_yolo` vérifie chaque ligne YOLO, ouvre les images pour connaître leur résolution et convertit les boîtes normalisées en pixels. `target_size=640` simule un redimensionnement proportionnel avec letterbox, comme lors d’un entraînement YOLO classique.

In [ ]:
if not INFACTORY_ROOT.is_dir():
    raise FileNotFoundError("Infactory n’est pas téléchargé. Voir download_hf.py.")

infactory = audit(
    INFACTORY_ROOT,
    missing_label_is_negative=True,
    target_size=640,
)
ball = infactory["class_metrics"][0]
ball_640 = ball["letterbox_640_px"]

display(Markdown(f"""
| Mesure | Valeur |
|---|---:|
| Images | {infactory['images']:,} |
| Images avec une boîte ballon valide | {ball['images_with_class']:,} |
| Images négatives explicites | {infactory['negative_images']:,} |
| Boîtes valides | {ball['boxes']:,} |
| Annotations invalides | {infactory['issues_count']:,} |
| Images avec plusieurs boîtes ballon | {ball['multi_box_images_count']:,} |
| Ballon médian en natif | {ball['width_px']['median']:.1f} × {ball['height_px']['median']:.1f} px |
| Ballon médian après letterbox 640 | {ball_640['width']['median']:.1f} × {ball_640['height']['median']:.1f} px |
"""))

print("Résolutions :", infactory["resolution_counts"])

### Pourquoi les séquences changent la manière de faire les splits

Deux frames voisines sont presque identiques. Elles ne doivent jamais être placées l’une dans le train et l’autre dans le test. L’unité de séparation sera donc l’UUID complet de la séquence, pas l’image.

In [ ]:
infactory_sequences = audit_sequences(INFACTORY_ROOT)
lengths = [row["frames"] for row in infactory_sequences["sequence_details"]]

display(Markdown(f"""
- **{infactory_sequences['sequences']} séquences** pour {infactory_sequences['parsed_frames']:,} frames ;
- longueur minimale / médiane / maximale : **{min(lengths)} / {infactory_sequences['sequence_length_frames']['median']:.0f} / {max(lengths)} frames** ;
- **{infactory_sequences['contiguous_sequences']} séquences continues**, sans numéro de frame manquant ;
- **{infactory_sequences['all_negative_sequences']} séquences entièrement négatives** ;
- **{infactory_sequences['sequences_with_invalid_annotations']} séquences** contenant des annotations invalides.
"""))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=15, color="#3b82f6", edgecolor="white")
ax.set(title="Longueur des séquences Infactory", xlabel="Nombre de frames", ylabel="Séquences")
plt.show()

### Inspection visuelle

Les cadres rouges ci-dessous sont lus directement depuis les fichiers YOLO. On affiche automatiquement un cas normal, un négatif, un cas multi-boîtes et un cas invalide. Le but n’est pas encore de corriger : il s’agit de comprendre ce que les statistiques cachent.

In [ ]:
def valid_boxes(image_path, dataset_root):
    label_path = label_path_for(image_path, dataset_root)
    if not label_path.exists():
        return []
    boxes = []
    with Image.open(image_path) as image:
        width, height = image.size
    for row in label_path.read_text().splitlines():
        fields = row.split()
        if len(fields) != 5:
            continue
        try:
            class_id = int(fields[0])
            x, y, box_width, box_height = map(float, fields[1:])
        except ValueError:
            continue
        if box_width <= 0 or box_height <= 0:
            continue
        boxes.append((
            class_id,
            (x - box_width / 2) * width,
            (y - box_height / 2) * height,
            (x + box_width / 2) * width,
            (y + box_height / 2) * height,
        ))
    return boxes

def annotated_image(image_path, dataset_root, class_ids=None):
    image = Image.open(image_path).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for class_id, left, top, right, bottom in valid_boxes(image_path, dataset_root):
        if class_ids is not None and class_id not in class_ids:
            continue
        color = "red" if class_id == 0 else "deepskyblue"
        line_width = 4 if class_id == 0 else 2
        drawing.rectangle((left, top, right, bottom), outline=color, width=line_width)
    return image

positive = next(
    path for path in sorted((INFACTORY_ROOT / "visible").glob("*.jpg"))
    if len(valid_boxes(path, INFACTORY_ROOT)) == 1
)
negative = next((INFACTORY_ROOT / "not_visible").glob("*.jpg"))
multi = INFACTORY_ROOT / ball["multi_box_image_examples"][0]
invalid_label = infactory["issues"][0].split(": ", 1)[1].rsplit(":", 1)[0]
invalid = (INFACTORY_ROOT / invalid_label).with_suffix(".jpg")

examples = [
    ("Une boîte valide", positive),
    ("Négatif : aucun ballon annoté", negative),
    ("Plusieurs boîtes : revue nécessaire", multi),
    ("Boîte de surface nulle", invalid),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, (title, image_path) in zip(axes.flat, examples):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

### Infactory : du plus petit au plus gros ballon

Une médiane seule cache des situations très différentes. Cette galerie sélectionne automatiquement six images selon la surface de leur boîte : minimum, 10e percentile, médiane, 90e percentile et maximum. Elle permet notamment de voir que les très grosses boîtes proviennent souvent de gros plans TV, peu représentatifs d’une caméra amateur fixe.

In [ ]:
infactory_single_boxes = []
for image_path in sorted((INFACTORY_ROOT / "visible").glob("*.jpg")):
    boxes = valid_boxes(image_path, INFACTORY_ROOT)
    if len(boxes) != 1:
        continue
    _, left, top, right, bottom = boxes[0]
    area = (right - left) * (bottom - top)
    infactory_single_boxes.append((area, image_path, right - left, bottom - top))

infactory_single_boxes.sort(key=lambda row: row[0])
quantiles = [0.00, 0.10, 0.25, 0.50, 0.90, 1.00]
selected = [
    infactory_single_boxes[round((len(infactory_single_boxes) - 1) * quantile)]
    for quantile in quantiles
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, quantile, (_, image_path, width, height) in zip(axes.flat, quantiles, selected):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT, class_ids={0}))
    axis.set_title(f"quantile {quantile:.0%} — {width:.1f} × {height:.1f} px")
    axis.axis("off")
plt.tight_layout()
plt.show()

### Infactory : une séquence dans le temps

Les images suivantes viennent toutes du même UUID. `visible` signifie qu’une boîte valide existe ; `négatif` signifie que le ballon est absent, caché ou hors champ. Cette vue explique pourquoi nous voulons conserver les séquences pour le tracking, mais aussi pourquoi une séquence entière doit rester dans un seul split.

In [ ]:
mixed_sequences = [
    row for row in infactory_sequences["sequence_details"]
    if row["positive_frames"] > 0 and row["negative_frames"] > 0
]
sequence = max(mixed_sequences, key=lambda row: row["frames"])
sequence_id = sequence["sequence"]
frame_numbers = [
    round(index * sequence["last_frame"] / 7) for index in range(8)
]

sequence_samples = []
for frame_number in frame_numbers:
    filename = f"{sequence_id}_{frame_number:06d}.jpg"
    visible_path = INFACTORY_ROOT / "visible" / filename
    negative_path = INFACTORY_ROOT / "not_visible" / filename
    image_path = visible_path if visible_path.exists() else negative_path
    state = "visible" if visible_path.exists() else "négatif"
    sequence_samples.append((frame_number, state, image_path))

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for axis, (frame_number, state, image_path) in zip(axes.flat, sequence_samples):
    axis.imshow(annotated_image(image_path, INFACTORY_ROOT, class_ids={0}))
    axis.set_title(f"frame {frame_number} — {state}")
    axis.axis("off")
fig.suptitle(f"Séquence {sequence_id[:8]}… ({sequence['frames']} frames)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. SoccerEye : intérêt et limites

SoccerEye contient deux classes : `0 = ballon`, `1 = personne`. Pour PitchLens, une image qui ne contient que des personnes est un négatif valable pour la classe ballon. L’audit de fuite analyse également les noms Roboflow et les empreintes SHA-256.

In [ ]:
if not SOCCEREYE_ROOT.is_dir():
    raise FileNotFoundError("SoccerEye n’est pas extrait dans data/interim.")

soccereye = audit(SOCCEREYE_ROOT, target_size=640)
soccereye_leakage = audit_splits(SOCCEREYE_ROOT)
soccer_ball = soccereye["class_metrics"][0]
soccer_ball_640 = soccer_ball["letterbox_640_px"]

display(Markdown(f"""
| Mesure | Valeur |
|---|---:|
| Images | {soccereye['images']:,} |
| Images avec ballon | {soccer_ball['images_with_class']:,} |
| Images sans ballon | {soccer_ball['images_without_class']:,} |
| Boîtes ballon | {soccer_ball['boxes']:,} |
| Ballon médian après letterbox 640 | {soccer_ball_640['width']['median']:.1f} × {soccer_ball_640['height']['median']:.1f} px |
| Noms d’image parent présents dans plusieurs splits | {soccereye_leakage['parent_names_across_splits']:,} |
| Préfixes source présents dans plusieurs splits | {soccereye_leakage['source_prefixes_across_splits']:,} / {soccereye_leakage['unique_source_prefixes']:,} |
| Fichiers strictement identiques entre splits | {soccereye_leakage['exact_file_hashes_across_splits']:,} |
"""))

### SoccerEye : inspection des ballons annotés

La classe personne est volontairement masquée ici pour rendre le ballon lisible. Les six exemples sont choisis du plus petit au plus grand ballon. Le titre indique aussi le split et la résolution d’origine.

In [ ]:
soccer_images = sorted(
    path for path in SOCCEREYE_ROOT.rglob("*.jpg") if "images" in path.parts
)
soccer_ball_images = []
for image_path in soccer_images:
    ball_boxes = [box for box in valid_boxes(image_path, SOCCEREYE_ROOT) if box[0] == 0]
    if not ball_boxes:
        continue
    largest_ball = max(
        ball_boxes,
        key=lambda box: (box[3] - box[1]) * (box[4] - box[2]),
    )
    _, left, top, right, bottom = largest_ball
    soccer_ball_images.append(
        ((right - left) * (bottom - top), image_path, right - left, bottom - top)
    )

soccer_ball_images.sort(key=lambda row: row[0])
quantiles = [0.00, 0.10, 0.25, 0.50, 0.90, 1.00]
selected = [
    soccer_ball_images[round((len(soccer_ball_images) - 1) * quantile)]
    for quantile in quantiles
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for axis, quantile, (_, image_path, width, height) in zip(axes.flat, quantiles, selected):
    with Image.open(image_path) as image:
        resolution = f"{image.width}×{image.height}"
    split = next(part for part in image_path.parts if part in {"train", "valid", "test"})
    axis.imshow(annotated_image(image_path, SOCCEREYE_ROOT, class_ids={0}))
    axis.set_title(
        f"q{quantile:.0%} — {split} — {resolution}\nballon {width:.1f} × {height:.1f} px"
    )
    axis.axis("off")
plt.tight_layout()
plt.show()

### SoccerEye : voir la fuite entre les splits

Chaque ligne ci-dessous montre deux variantes portant le même nom parent Roboflow mais placées dans deux splits différents. Elles ne sont pas identiques octet par octet, mais proviennent de la même image parente. Une évaluation construite ainsi peut récompenser la mémorisation plutôt que la généralisation.

In [ ]:
leakage_pairs = []
for parent in soccereye_leakage["parent_split_examples"]:
    matches = [path for path in soccer_images if parent_name(path) == parent]
    by_split = {}
    for image_path in matches:
        split = next(part for part in image_path.parts if part in {"train", "valid", "test"})
        by_split.setdefault(split, image_path)
    if len(by_split) >= 2:
        first_two = list(sorted(by_split.items()))[:2]
        leakage_pairs.append((parent, first_two))
    if len(leakage_pairs) == 3:
        break

fig, axes = plt.subplots(len(leakage_pairs), 2, figsize=(13, 5 * len(leakage_pairs)))
for row_axes, (parent, pair) in zip(axes, leakage_pairs):
    for axis, (split, image_path) in zip(row_axes, pair):
        axis.imshow(annotated_image(image_path, SOCCEREYE_ROOT, class_ids={0}))
        axis.set_title(f"parent {parent} — split {split}")
        axis.axis("off")
plt.tight_layout()
plt.show()

## 4. Comparaison à la taille d’entrée mobile

Comparer les pixels natifs serait trompeur puisque les résolutions diffèrent. Le graphique utilise donc uniquement les dimensions après letterbox à `640 × 640`.

In [ ]:
names = ["Infactory", "SoccerEye"]
widths = [ball_640["width"]["median"], soccer_ball_640["width"]["median"]]
heights = [ball_640["height"]["median"], soccer_ball_640["height"]["median"]]
positions = range(len(names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([index - 0.18 for index in positions], widths, width=0.36, label="largeur")
axes[0].bar([index + 0.18 for index in positions], heights, width=0.36, label="hauteur")
axes[0].set_xticks(list(positions), names)
axes[0].set_ylabel("Pixels à l’entrée 640")
axes[0].set_title("Taille médiane du ballon")
axes[0].legend()

with_ball = [ball["images_with_class"], soccer_ball["images_with_class"]]
without_ball = [
    infactory["images"] - ball["images_with_class"],
    soccer_ball["images_without_class"],
]
axes[1].bar(names, with_ball, label="avec ballon valide")
axes[1].bar(names, without_ball, bottom=with_ball, label="sans ballon valide")
axes[1].set_ylabel("Images")
axes[1].set_title("Présence du ballon")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Conclusion et décision

### Ce que les données permettent aujourd’hui

- **Infactory** est le meilleur choix pour construire une première baseline : davantage de ballons, séquences temporelles et vrais négatifs.
- **SoccerEye** est un complément intéressant pour les très petits ballons, mais pas une base d’évaluation fiable avec ses splits actuels.
- Les deux sources proviennent surtout de retransmissions professionnelles. Elles ne prouvent pas encore qu’un iPhone fixe suivra correctement un match amateur.

### Prochaine étape

Construire `data/processed/v001` uniquement à partir d’Infactory :

1. revoir les 75 annotations invalides ;
2. revoir les 63 images multi-boîtes ;
3. créer les splits par UUID complet ;
4. enregistrer la composition et les statistiques de `v001` ;
5. entraîner ensuite une première baseline nano.

SoccerEye sera testé séparément dans une seconde version. Cette séparation permettra de mesurer son apport réel au lieu de supposer que davantage de données améliore forcément le modèle.